# Analytics & Temporal Reasoning

KPIs, entities, hierarchies, comparisons, formatting, and time-aware analysis.

**Rollup: 47/47 cases passed** across 6 required capabilities, est. cost $0.0542, 2915s total.

This is an executive-level summary over already-captured real-LLM results -- see `docs/CAPABILITY_MAPPING.md` for the full 25-capability table, and `notebooks/capabilities/<NN>_<slug>/demo.ipynb` for every case in full detail per capability.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 14. Standardized formatting, including markdown tables and unit-aware presentation

7/7 passed — $0.0081, 535s

In [ ]:
_ = ask("Give me North America's revenue, volume, and EBITDA margin for Q1 2024", '01')
_ = ask('Compare revenue between North America and EMEA in 2025', '02')

01
USER: Give me North America's revenue, volume, and EBITDA margin for Q1 2024

[intent=data_query | sub_agents=['structured']]

AGENT: ## North America — Q1 2024

| KPI | Value |
| --- | --- |
| Revenue | **$3,593 million** |
| Volume | **21,353 thousand hL** |
| EBITDA Margin | **31.3%** |

At $3,593m revenue and 21,353 khl volume, that implies roughly **$168 per hL** of revenue in the quarter.

Note: the EBITDA margin figure is flagged as computed in the source data, so it's derived rather than a directly reported line item.

Follow-ups you might find useful:
- How does North America's Q1 2024 revenue compare to the same quarter last year?
- What drove the volume trend — beer vs. beyond-beer brands?

02
USER: Compare revenue between North America and EMEA in 2025

[intent=comparison | sub_agents=['structured']]

AGENT: ## Revenue comparison — North America vs. EMEA (2025)

Based on the 2025 quarterly-grain revenue data (summed across the four quarters):

| Zone | Revenue (USD m, 20

## 15. Temporal reasoning for current, historical, and comparative period analysis

8/8 passed — $0.0101, 614s

In [ ]:
_ = ask("What is AB InBev's most recent quarterly revenue?", '01')
_ = ask("What was North America's revenue in Q1 2024?", '02')

01
USER: What is AB InBev's most recent quarterly revenue?

[intent=data_query | sub_agents=['structured']]

AGENT: AB InBev's most recent reported quarterly revenue for Global is **$15,384 million** in **Q4 2025** (period label: Q4 2025). [Source: structured KPI data]

If useful, I can check how that compares to the prior quarter or the same quarter a year earlier, or break it down by region/segment.

02
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

## 18. Metadata queries

7/7 passed — $0.0037, 251s

In [ ]:
_ = ask('What KPIs do you track?', '01')
_ = ask('What zones can I ask about?', '02')

01
USER: What KPIs do you track?

[intent=metadata_discovery | sub_agents=[]]

AGENT: **Available data**:

KPIs: Revenue (USD million), Volume (thousand hL), Normalized EBITDA (USD million), EBITDA Margin (% (computed)), Organic Revenue Growth (%), Net Profit (USD million)

Structured (SQL) grain: zone x quarter, Q1 2024-Q4 2025; zone x year and company-wide (Global) x year, FY2022-FY2025. No brand-level or country-level structured rows -- AB InBev doesn't disclose that granularity publicly.

Zones: North America, Middle Americas, South America, EMEA, Asia Pacific
Countries (roll up to their zone): United States (North America), Canada (North America), Mexico (Middle Americas), Colombia (Middle Americas), Peru (Middle Americas), Ecuador (Middle Americas), Brazil (South America), Argentina (South America), United Kingdom (EMEA), Netherlands (EMEA), France (EMEA), Italy (EMEA), South Africa (EMEA), Nigeria (EMEA), China (Asia Pacific), South Korea (Asia Pacific)
Brands (document/qualitat

## 19. Multiple KPIs, entities, dimensions, and hierarchical business structures

10/10 passed — $0.0128, 673s

In [ ]:
_ = ask("What was North America's revenue in Q1 2024?", '01')
_ = ask("What was EMEA's volume in Q2 2025?", '02')

01
USER: What was North America's revenue in Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America's revenue in **Q1 2024** was **$3,593 million (USD)** [structured KPI data].

Want to compare this with Q1 2025 or break it down by brand/segment?

02
USER: What was EMEA's volume in Q2 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA’s volume in Q2 2025 was **24,172.0 thousand hectoliters (hL)**.

If you’d like, I can help pull the year-over-year change or a country/brand breakdown for the same period.

## 20. Analytical comparisons across KPIs, entities, periods, and business domains

8/8 passed — $0.0151, 679s

In [ ]:
_ = ask('Compare revenue between North America and EMEA in 2025', '01')
_ = ask("How did North America's revenue in 2025 compare to 2024?", '02')

01
USER: Compare revenue between North America and EMEA in 2025

[intent=comparison | sub_agents=['structured']]

AGENT: ## Revenue comparison — North America vs. EMEA (2025)

| Zone | Revenue (USD m, 2025) |
| --- | --- |
| North America | 14,208.0 |
| EMEA | 9,502.0 |

**Key takeaways**

- **North America generated more revenue than EMEA in 2025**: 14,208 m USD vs. 9,502 m USD — a difference of **4,706 m USD** in North America's favor.

**Data notes / caveats**

- These figures are the summed quarterly-grain revenue records for 2025 for the two requested zones only; other zones (e.g., Latin America, Asia Pacific) were not part of the query, so this is not a company-wide revenue total.
- The evidence contains only annual zone totals for 2025 — no quarterly split and no prior-year comparison — so I can't tell whether this gap was consistent across the year or how it changed versus 2024.

**Suggested follow-ups**

- View the **quarterly 2025 breakdown** for each zone to see when the gap

## 22. Metadata discovery for available KPIs, dimensions, periods, and datasets

7/7 passed — $0.0045, 163s

In [ ]:
_ = ask('What metrics are available and in what units?', '01')
_ = ask('List all the zones/regions you have data for', '02')

01
USER: What metrics are available and in what units?

[intent=metadata_discovery | sub_agents=[]]

AGENT: **Available data**:

KPIs: Revenue (USD million), Volume (thousand hL), Normalized EBITDA (USD million), EBITDA Margin (% (computed)), Organic Revenue Growth (%), Net Profit (USD million)

Structured (SQL) grain: zone x quarter, Q1 2024-Q4 2025; zone x year and company-wide (Global) x year, FY2022-FY2025. No brand-level or country-level structured rows -- AB InBev doesn't disclose that granularity publicly.

Zones: North America, Middle Americas, South America, EMEA, Asia Pacific
Countries (roll up to their zone): United States (North America), Canada (North America), Mexico (Middle Americas), Colombia (Middle Americas), Peru (Middle Americas), Ecuador (Middle Americas), Brazil (South America), Argentina (South America), United Kingdom (EMEA), Netherlands (EMEA), France (EMEA), Italy (EMEA), South Africa (EMEA), Nigeria (EMEA), China (Asia Pacific), South Korea (Asia Pacific)
Bra